# 6. SHAP Analysis

SHAP (SHapley Additive exPlanations) interpretability analysis using TreeExplainer on the trained Random Forest model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

## 1. Load Data & Train RF / RF

In [ ]:
# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent
df = pd.read_csv(PROJECT_ROOT / 'data' / 'samples' / 'dataset_training.csv')
FACTORS = ['elevation', 'slope', 'distance_to_river', 'distance_to_coast',
    'land_cover', 'soil_type', 'ndvi', 'rainfall']

X = df[FACTORS].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
print(f'RF trained. Test accuracy: {rf.score(X_test, y_test):.3f}')

## 2. Compute SHAP Values / SHAP

TreeExplainer is used for efficient exact SHAP computation on tree-based models.

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Handle different SHAP version outputs
if isinstance(shap_values, list):
    shap_vals = shap_values[1] # class 1 (flood)
elif len(shap_values.shape) == 3:
    shap_vals = shap_values[:, :, 1]
else:
    shap_vals = shap_values

print(f'SHAP values shape: {shap_vals.shape}')
print(f'X_test shape: {X_test.shape}')

## 3. SHAP Beeswarm Plot / SHAP

Shows the distribution of SHAP values for each feature across all test samples.
Positive SHAP values push predictions toward flood (class 1).

In [ ]:
plt.figure()
shap.summary_plot(shap_vals, X_test, feature_names=FACTORS, show=False)
plt.tight_layout()
# plt.savefig(PROJECT_ROOT / 'outputs' / 'shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## 4. SHAP Bar Plot (Mean |SHAP|) / SHAP

In [ ]:
plt.figure()
shap.summary_plot(shap_vals, X_test, feature_names=FACTORS,
    plot_type='bar', show=False)
plt.tight_layout()
# plt.savefig(PROJECT_ROOT / 'outputs' / 'shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## 5. SHAP Values Table / SHAP

Mean absolute SHAP values for each feature, ranked by importance.

In [ ]:
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': FACTORS,
    'Mean|SHAP|': [round(v, 4) for v in mean_abs_shap]
}).sort_values('Mean|SHAP|', ascending=False)

shap_importance.to_csv(PROJECT_ROOT / 'outputs' / 'shap_importance.csv', index=False)
print(shap_importance.to_string(index=False))